# Create HAL configs and shutters for the bits rounds -- variable z per FOV

Reads the per-FOV z table (`metadata/z_per_fov_table.csv`, from
`after_imaging/multi_z_margin_export.ipynb`) and groups FOVs into a few z-depth
**tiers**. Each tier gets its own HAL config + shutter, so thin-tissue FOVs
take a shorter movie than thick ones.

**On the microscope:** each positions-file line gets a 3rd column naming the
tier's HAL config, and Dave switches HAL parameters per FOV. Load every tier's
HAL config into HAL before the round starts (by hand, `N_TIERS` files).

**Files:** tier HAL configs and shutters go together in `SAMPLE_DIR/multi_z/`
(HAL finds shutters in the config's own folder). Frame tables go to
`metadata/`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/multi_z/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config, power_dict_to_channel_list,
    frame_table_filename, shutter_filename, hal_config_filename,
)
from MERci.acquisition.display import print_frame_table, display_xml
from MERci.plots.experiment_plots       import visualize_shutter_sequence
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"     # regular (cells/transit) configs -- unaffected
METADATA_DIR = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MULTI_Z_DIR  = SAMPLE_DIR / "multi_z"      # tier hal_configs + shutters
MULTI_Z_DIR.mkdir(parents=True, exist_ok=True)

# Positions files are named with POSITIONS_TAG.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

MICROSCOPE = "ST2"   # must match notebook 01

_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}")
HAL_TEMPLATE = _hal_candidates[0]

FILE_TYPE     = ".zarr"
EXPOSURE_TIME = 0.15      # seconds -- must match notebook 01's bits round exposure

# Same POWER convention as notebook 01. ST2/MFX have no 750.
POWER = {
    650: 1.00,
    560: 1.00,
    488: 1.00,
    405: 1.00,
}
POWER_DEFAULT = 1.0

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"POSITIONS_TAG: {POSITIONS_TAG}")
print(f"MULTI_Z_DIR  : {MULTI_Z_DIR}")
print(f"HAL template : {HAL_TEMPLATE.name}")

## Bucket FOVs into z-depth tiers

`N_TIERS` stays small, since each tier is loaded into HAL by hand. FOVs are
split by quantile (same FOV count per tier), so a few very thick FOVs don't
put everything else in one tier. Each tier images the deepest `z_needed_um`
among its FOVs, so no FOV gets less depth than it needs.

In [ ]:
N_TIERS = 15   # how many discrete hal_config/shutter variants to generate
TIER_LABELS = [f"tier{i:02d}" for i in range(N_TIERS)]   # ascending depth order (tier00 = shallowest)

z_per_fov_path = METADATA_DIR / "z_per_fov_table.csv"
if not z_per_fov_path.exists():
    raise FileNotFoundError(f"{z_per_fov_path} not found -- run after_imaging/multi_z_margin_export.ipynb first.")
z_per_fov = pd.read_csv(z_per_fov_path)

z_needed = z_per_fov["z_needed_um"].dropna()
if len(z_needed) == 0:
    raise ValueError("No FOV has a z_needed_um value -- check after_imaging/multi_z_margin_export.ipynb's THRESHOLD/results.")

quantile_edges = np.unique(np.quantile(z_needed, np.linspace(0, 1, N_TIERS + 1)))
if len(quantile_edges) - 1 < N_TIERS:
    print(f"WARNING: only {len(quantile_edges) - 1} distinct z_needed_um quantile edge(s) "
          f"(requested {N_TIERS} tiers) -- some FOVs share an identical z_needed_um. "
          f"Reducing to {len(quantile_edges) - 1} tier(s).")
    TIER_LABELS = TIER_LABELS[: len(quantile_edges) - 1]

z_per_fov["tier"] = pd.cut(z_per_fov["z_needed_um"], bins=quantile_edges,
                            labels=TIER_LABELS, include_lowest=True)

# FOVs with no z_needed_um (no signal in measure_tissue_thickness) get the
# shallowest tier: they were tissue-free border FOVs. Check the
# multi_z_margin_export mosaic before relying on this for a new experiment.
n_no_signal = int(z_per_fov["tier"].isna().sum())
z_per_fov["tier"] = z_per_fov["tier"].astype(object).where(z_per_fov["tier"].notna(), TIER_LABELS[0])

tier_depth_um = z_per_fov.groupby("tier")["z_needed_um"].max().to_dict()
for tier in TIER_LABELS:
    if tier not in tier_depth_um or pd.isna(tier_depth_um.get(tier)):
        tier_depth_um[tier] = z_per_fov.loc[z_per_fov["tier"] == tier, "z_needed_um"].max()

print(z_per_fov["tier"].value_counts().reindex(TIER_LABELS))
print()
for tier in TIER_LABELS:
    print(f"  {tier:8s}: up to {tier_depth_um[tier]:.1f} um")
if n_no_signal:
    print(f"\n{n_no_signal} FOV(s) had no detected signal in after_imaging/measure_thickness/measure_tissue_thickness.ipynb -- "
          f"assigned the shallowest tier ({TIER_LABELS[0]}) as confirmed tissue-free "
          f"(verified via after_imaging/multi_z_margin_export.ipynb's section 12 mosaic).")

## Generate one hal_config + shutter per tier

Same imaging sequence as notebook 01's bits round (keep them in sync); only
`z_max` changes per tier.

In [ ]:
z_bead    = 0
z_min     = 0.5
z_step    = 0.5
bead_seq  = [488]
color_seq = [650, 560]
end_seq   = [488]

SCAN_MODE     = "interleaved"
Z_RETURN_MODE = "progressive"
RETURN_STEP   = 5

tier_hal_configs = {}   # tier label -> {hal_stem, n_frames, z_max_um}
for tier in TIER_LABELS:
    z_max = float(tier_depth_um[tier])
    z_pos = np.arange(z_min, z_max + 1, z_step)

    frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                                   microscope=MICROSCOPE, scan_mode=SCAN_MODE,
                                   z_return_mode=Z_RETURN_MODE, return_step=RETURN_STEP)
    name = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)

    ft_path = METADATA_DIR / frame_table_filename("bits", name, tier=tier)
    frame_table.to_csv(ft_path)

    shutter_name = shutter_filename("bits", name, tier=tier)
    # Shutter power is always POWER_DEFAULT; POWER sets <default_power> below.
    create_shutter_file(frame_table, MULTI_Z_DIR / shutter_name,
                         default_power=POWER_DEFAULT)

    hal_output = MULTI_Z_DIR / hal_config_filename(MICROSCOPE, "bits", name, tier=tier)
    create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                      default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                      file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)

    tier_hal_configs[tier] = {
        "hal_stem": hal_output.stem, "n_frames": len(frame_table), "z_max_um": z_max,
    }
    print(f"[{tier:8s}] z_max={z_max:6.1f} um, {len(frame_table):4d} frames -> {hal_output.name}")

# For notebook 05. The deepest tier (last row) is round_info's hal_config.
z_tiers_df = pd.DataFrame([{"tier": t, **info} for t, info in tier_hal_configs.items()])
z_tiers_df = z_tiers_df.sort_values("z_max_um").reset_index(drop=True)
z_tiers_path = METADATA_DIR / "z_tiers.csv"
z_tiers_df.to_csv(z_tiers_path, index=False)
print(f"\nSaved: {z_tiers_path}")
print(z_tiers_df.to_string(index=False))

## Add the per-FOV tier column to the bits positions file(s)

Dave's patched `v2Generator.py` (`misc/dave_multi_z/`) reads an optional 3rd
column per positions line: the HAL config for that FOV. The positions files
are rewritten in place (re-running replaces the column). The cells round uses
the same files and ignores the column.

FOVs are matched to `measure_tissue_thickness`'s `fov_id`s by nearest stage
position, not line order.

In [ ]:
from MERci.common.metadata import ExperimentMetadata
from MERci.common.config   import ExperimentConfig

# Same metadata as measure_tissue_thickness, so fov_id <-> (x, y) matches.
_config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = METADATA_DIR,
    analysis_dir   = SAMPLE_DIR / "analysis",
    round_info_csv = METADATA_DIR / "round_info.csv",
    positions_txt  = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt",
)
_meta = ExperimentMetadata.load(_config.round_info_csv, _config.positions_txt, _config.data_dir)

fov_ids   = np.array(list(_meta.fovs.keys()))
positions = np.array([_meta.fovs[f].position for f in fov_ids])   # (N, 2)

fov_id_to_tier_stem = dict(zip(
    z_per_fov["fov_id"], z_per_fov["tier"].map(lambda t: tier_hal_configs[t]["hal_stem"]),
))


def nearest_fov_id(x, y, max_distance_um=1.0):
    """Match a positions-file (x, y) to after_imaging/measure_thickness/measure_tissue_thickness.ipynb's fov_id by nearest stage
    position -- robust to positions files not enumerating FOVs in fov_id order
    (true for a multi-boundary layout's per-segment files). Raises if the
    closest match is implausibly far, since that means this positions file
    doesn't actually belong to this experiment's metadata."""
    d = np.hypot(positions[:, 0] - x, positions[:, 1] - y)
    i = int(np.argmin(d))
    if d[i] > max_distance_um:
        raise ValueError(
            f"No FOV within {max_distance_um} um of ({x}, {y}) -- closest is "
            f"{d[i]:.2f} um away (fov_id {fov_ids[i]}). Positions file/metadata mismatch?"
        )
    return int(fov_ids[i])


TRANSIT_TOKEN = "transit"   # transit positions files are not tiered
positions_files = sorted(
    p for p in POSITIONS_DIR.glob(f"positions_{POSITIONS_TAG}*.txt")
    if TRANSIT_TOKEN not in p.name
)
print(f"Positions file(s) to tag with tier column: {[p.name for p in positions_files]}")

for pfile in positions_files:
    lines = pfile.read_text().splitlines()
    new_lines = []
    n_tagged = 0
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            new_lines.append(line)
            continue
        fields = stripped.split(",")
        x, y = float(fields[0]), float(fields[1])
        fov_id = nearest_fov_id(x, y)
        tier_stem = fov_id_to_tier_stem.get(fov_id)
        if tier_stem is None:
            raise ValueError(f"fov_id {fov_id} ({pfile.name}) has no tier assignment -- "
                              f"check z_per_fov_table.csv/after_imaging/multi_z_margin_export.ipynb.")
        new_lines.append(f"{x},{y},{tier_stem}")
        n_tagged += 1
    pfile.write_text("\n".join(new_lines) + "\n")
    print(f"  {pfile.name}: {n_tagged} FOV(s) tagged with a tier hal_config")